# Stage 1 · 任务学习
复用已训练 OpenJev-4B，BF16 骨干 + LoRA，FP32 原决策头与交互层。先实测单卡 pilot，再开始正式训练。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Reuse the prepared data; RUN_FULL_STAGE1 below controls training size.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    # BF16 + LoRA does not use torchao quantization. Colab's old optional
    # torchao package makes PEFT 0.19.1 fail even on ordinary Linear layers.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
%pip uninstall -y torchao

In [4]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [5]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3, 'GiB')
# Stream both output channels into the notebook and preserve the full traceback.
log_path = ROOT / 'qwen_lora_check.log'
cmd = [sys.executable, '-u', 'scripts/check_qwen_lora.py', '--device', 'cuda',
       '--compare-batching', '--output', str(ROOT/'qwen_lora_check.json')]
print('Tiny random model check (no downloads). Log:', log_path, flush=True)
with log_path.open('w', encoding='utf-8') as log:
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
if returncode:
    raise RuntimeError(f'Qwen LoRA check failed (exit {returncode}); see traceback above or {log_path}')


NVIDIA A100-SXM4-40GB 39.4935302734375 GiB
Tiny random model check (no downloads). Log: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/qwen_lora_check.log
[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
{'device': 'cuda', 'model': 'random tiny Qwen3.5, mixed linear/full attention', 'lor

In [6]:
# Reuse full prepared data. This switch controls training, not the data directory.
import copy, importlib, torch
from sa_jev import model, training, experiments
for module in (model, training, experiments):
    importlib.reload(module)
RUN_FULL_STAGE1 = False
PROBE_STEPS = 6
TRAIN_SEEDS = [17]  # After seed 17 completes, run [29, 43] for the remaining formal seeds.
assert torch.cuda.is_available(), 'Select an A100 GPU runtime'
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
config['model']['branch_microbatch'] = 2
config['model']['decision_batch_size'] = 8
config['training']['validate_every'] = 500
config['training']['save_every'] = 25
config['training']['memory_limit_gib'] = min(38.0, GPU_GIB - 2.0)
# accumulation is the effective number of states per update (legacy name).
# Two states run together; four microbatches make one 8-state update.
config['training']['accumulation'] = 8
config['training']['state_batch_size'] = 2
print('Data ROOT:', ROOT, flush=True)
print('GPU:', torch.cuda.get_device_name(0), 'capacity GiB:', round(GPU_GIB, 2), flush=True)
print('Native BF16:', torch.cuda.is_bf16_supported(including_emulation=False), flush=True)
print('Memory stop threshold GiB:', config['training']['memory_limit_gib'], flush=True)
print('Concurrent states:', config['training']['state_batch_size'], 'Effective batch states:', config['training']['accumulation'], flush=True)
print('Branch microbatch:', config['model']['branch_microbatch'], 'Decision batch:', config['model']['decision_batch_size'], flush=True)
print('Mode:', 'formal Stage 1' if RUN_FULL_STAGE1 else '6-step batching probe in a separate directory', flush=True)


Data ROOT: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full
GPU: NVIDIA A100-SXM4-40GB capacity GiB: 39.49
Native BF16: True
Memory stop threshold GiB: 37.4935302734375
Mode: 20-step probe in a separate directory


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from sa_jev.experiments import train_variant
stage1_config = copy.deepcopy(config)
if not RUN_FULL_STAGE1:
    stage1_config['training']['validate_every'] = 10
    stage1_config['training']['save_every'] = 10
probe_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
for seed in (TRAIN_SEEDS if RUN_FULL_STAGE1 else [17]):
    probe_output = None if RUN_FULL_STAGE1 else ROOT / 'runs' / 'stage1_probe' / probe_id / f'seed_{seed}'
    history = train_variant(
        ROOT, stage1_config, 'stage1', seed,
        max_steps=None if RUN_FULL_STAGE1 else PROBE_STEPS,
        validation_limit=None if RUN_FULL_STAGE1 else 6,
        output_directory=probe_output,
    )
    print('Finished steps:', len(history),
          'peak reserved GiB:', max(row['peak_reserved_gib'] for row in history),
          'total valid rank observations:', sum(row['valid_rank'] for row in history), flush=True)
    from statistics import median
    timed = [row['step_seconds'] for row in history[1:5] if 'step_seconds' in row]
    if timed:
        print('Median training seconds/step (steps 2-5, excluding validation/save):', median(timed), flush=True)
    if probe_output is not None:
        print('Probe saved separately:', probe_output, flush=True)


[train] Reading /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/data/train_states.jsonl and validation states...
[train] Loaded 36000 training states, 6 validation states; valid rank train=23994, val=0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[OpenJev] Load backbone/head/tokenizer; target=cuda, dtype=bfloat16: exited after 13.2s
[OpenJev] Model loaded on cuda; configuring decision branches
[OpenJev] Delta kernel=fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule; convolution=torch
[train] Stage 1 run; progress below, output=/content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/runs/stage1_probe/20261112T100722Z_4f1e28/seed_17: starting
[train] GPU total=39.49 GiB; observed-memory stop threshold=37.49 GiB. This check does not prevent an allocation from raising CUDA OOM.
[train] Binding data/config: train=36000, validation=6, stage=1
[train] 20 optimizer steps; accumulation=8; validate_every=10, save_every=10; memory_limit=37.49 GiB; output=/content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/runs/stage1_probe/20261112T100722Z_4f1e28/seed_17
[train] START step 1/20, epoch=0, states=8
[train] step 1/20, state 1/8: forward/backward K=13, id=musique:triple_io__162189_38001_35312
[train] Stage 1 run; progress below, output=/c

检查 training.csv/PNG/PDF、验证 loss、实际显存和 ETA。4GB 本地 tiny 测试不能代替 4B A100 测试；OOM 时降低 branch_microbatch，不切分 softmax，不截断证据，不改变标签。恢复训练使用 train_variant(..., resume=checkpoint_path)。